In [2]:
import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.decomposition import PCA
import time
import shap
import matplotlib.pyplot as plt

# Function to extract MFCCs from audio files
def extract_mfccs(file_path):
    try:
        y, sr = librosa.load(file_path, duration=1)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=60)
        return np.mean(mfccs.T, axis=0)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Data Loading and Feature Extraction
male_folder = "/home/feliciano/adult_lobsters"
female_folder = "/home/feliciano/juvenile_lobsters"

data = []
labels = []

print("Loading audio files and extracting MFCCs...")
for filename in os.listdir(male_folder):
    file_path = os.path.join(male_folder, filename)
    if os.path.isfile(file_path) and filename.lower().endswith(('.wav', '.mp3', '.ogg', '.flac')):
        mfccs = extract_mfccs(file_path)
        if mfccs is not None:
            data.append(mfccs)
            labels.append("adult")

for filename in os.listdir(female_folder):
    file_path = os.path.join(female_folder, filename)
    if os.path.isfile(file_path) and filename.lower().endswith(('.wav', '.mp3', '.ogg', '.flac')):
        mfccs = extract_mfccs(file_path)
        if mfccs is not None:
            data.append(mfccs)
            labels.append("juvenile")

if not data:
    print("No audio files found in the specified folders. Please check your paths and file types.")
    exit()

X = np.array(data)
y = np.array(labels)

print(f"Successfully loaded {len(data)} audio samples.")

# Label Encoding and Data Scaling
le = LabelEncoder()
y = le.fit_transform(y)
class_names = le.classes_
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data before PCA to avoid data leakage in Grid Search
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# --- Apply PCA and Grid Search ---

print("\nPerforming GridSearchCV for PCA...")
pca_param_grid = {'n_components': [None, 10, 20, 30, 40]}
pca = PCA(random_state=42)
grid_search_pca = GridSearchCV(pca, pca_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search_pca.fit(X_train, y_train)

best_n_components = grid_search_pca.best_params_['n_components']
print(f"\nBest number of PCA components: {best_n_components}")

pca_final = PCA(n_components=best_n_components, random_state=42)
X_train_pca = pca_final.fit_transform(X_train)
X_test_pca = pca_final.transform(X_test)

print(f"Original feature shape (scaled): {X_train.shape}")
print(f"Shape after PCA: {X_train_pca.shape}")
if best_n_components is not None:
    print(f"Explained variance ratio of the first {best_n_components} components: {pca_final.explained_variance_ratio_}")
    print(f"Total explained variance: {sum(pca_final.explained_variance_ratio_):.4f}")
else:
    print("PCA: All components retained.")

# Define the parameter grid for MLPClassifier
mlp_param_grid = {
    'hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'sgd'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate': ['constant', 'adaptive'],
}

mlp_model = MLPClassifier(random_state=42, max_iter=500)

print("\nPerforming GridSearchCV for MLPClassifier...")
grid_search_mlp = GridSearchCV(mlp_model, mlp_param_grid, cv=5, scoring='accuracy', n_jobs=-1)

start_time_grid = time.time()
grid_search_mlp.fit(X_train_pca, y_train)
end_time_grid = time.time()
grid_search_time = (end_time_grid - start_time_grid)
print(f"\nMLP Grid Search Time: {grid_search_time:.2f} seconds")

best_params_mlp = grid_search_mlp.best_params_
best_score_mlp = grid_search_mlp.best_score_
print("\nBest MLP Parameters:", best_params_mlp)
print(f"Best Cross-Validation Accuracy (MLP with PCA): {best_score_mlp:.4f}")

best_mlp_model = grid_search_mlp.best_estimator_
start_time_inference = time.time()
y_pred_best = best_mlp_model.predict(X_test_pca)
inference_time = (time.time() - start_time_inference) * 1000

if hasattr(best_mlp_model, "predict_proba"):
    y_pred_proba_best = best_mlp_model.predict_proba(X_test_pca)[:, 1]
else:
    y_pred_proba_best = np.zeros_like(y_pred_best, dtype=float)

accuracy_best = accuracy_score(y_test, y_pred_best)
precision_best = precision_score(y_test, y_pred_best)
recall_best = recall_score(y_test, y_pred_best)
f1_best = f1_score(y_test, y_pred_best)

if hasattr(best_mlp_model, "predict_proba") and y_pred_proba_best.ndim > 0:
    auc_roc_best = roc_auc_score(y_test, y_pred_proba_best)
else:
    auc_roc_best = float('nan')

print("\nBest MLP Model Performance with PCA (on Test Set):")
print(f"  Accuracy: {accuracy_best:.4f}")
print(f"  Precision: {precision_best:.4f}")
print(f"  Recall: {recall_best:.4f}")
print(f"  F1-Score: {f1_best:.4f}")
print(f"  AUC-ROC: {auc_roc_best:.4f}" if not np.isnan(auc_roc_best) else "  AUC-ROC: Not calculable")
print(f"  Inference Time (ms): {inference_time:.4f}")
print("\nNote: MAP-50 is not directly applicable for binary classification.")

# --- SHAP Implementation ---

print("\n--- Starting SHAP Explanation ---")

background_data = X_train_pca[np.random.choice(X_train_pca.shape[0], 100, replace=False)]

explainer = shap.KernelExplainer(best_mlp_model.predict_proba, background_data)

print("Calculating SHAP values for the test set... This may take some time.")
# Calculate SHAP values for the test set
# We need to create feature names for the PCA components
pca_feature_names = [f'PC_{i+1}' for i in range(X_test_pca.shape[1])]

shap_values = explainer.shap_values(X_test_pca)

positive_class_idx = np.where(le.classes_ == 'juvenile')[0][0] if 'juvenile' in le.classes_ else 1

print(f"Explaining predictions for class: {class_names[positive_class_idx]}")

# 1. SHAP Summary Plot (Global Feature Importance and Impact)
plt.figure(figsize=(10, 6))
# Pass feature_names to the summary_plot
shap.summary_plot(
    shap_values[positive_class_idx],
    X_test_pca, # Pass X_test_pca as features
    feature_names=pca_feature_names, # Use the generated PCA feature names
    plot_type="dot",
    show=False
)
plt.title(f"SHAP Summary Plot for {class_names[positive_class_idx]} class")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values[positive_class_idx],
    X_test_pca, # Pass X_test_pca as features
    feature_names=pca_feature_names, # Use the generated PCA feature names
    plot_type="bar",
    show=False
)
plt.title(f"SHAP Bar Plot (Feature Importance) for {class_names[positive_class_idx]} class")
plt.tight_layout()
plt.show()

# 2. Individual Force Plots (Explaining a Single Prediction)
instance_to_explain_idx = 0
print(f"\n--- Explaining prediction for Test Sample {instance_to_explain_idx} ---")
print(f"True Label: {class_names[y_test[instance_to_explain_idx]]}")
print(f"Predicted Label: {class_names[y_pred_best[instance_to_explain_idx]]}")
print(f"Predicted Probability ({class_names[positive_class_idx]}): {best_mlp_model.predict_proba(X_test_pca[instance_to_explain_idx:instance_to_explain_idx+1])[:, positive_class_idx][0]:.4f}")

shap.initjs()
shap.force_plot(
    explainer.expected_value[positive_class_idx],
    shap_values[positive_class_idx][instance_to_explain_idx],
    X_test_pca[instance_to_explain_idx],
    feature_names=pca_feature_names, # Use the generated PCA feature names here as well
    show=False
)
plt.title(f"SHAP Force Plot for Test Sample {instance_to_explain_idx} ({class_names[y_test[instance_to_explain_idx]]})")
plt.show()

Loading audio files and extracting MFCCs...
Successfully loaded 7307 audio samples.

Performing GridSearchCV for PCA...


/home/feliciano/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:993: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/feliciano/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 982, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/feliciano/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 253, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/feliciano/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 344, in _score
    response_method = _check_response_method(estimator, self._response_method)
                      


Best number of PCA components: None
Original feature shape (scaled): (5845, 60)
Shape after PCA: (5845, 60)
PCA: All components retained.

Performing GridSearchCV for MLPClassifier...

MLP Grid Search Time: 87.01 seconds

Best MLP Parameters: {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (128,), 'learning_rate': 'constant', 'solver': 'adam'}
Best Cross-Validation Accuracy (MLP with PCA): 0.9872

Best MLP Model Performance with PCA (on Test Set):
  Accuracy: 0.9911
  Precision: 0.9854
  Recall: 0.9875
  F1-Score: 0.9865
  AUC-ROC: 0.9995
  Inference Time (ms): 0.8795

Note: MAP-50 is not directly applicable for binary classification.

--- Starting SHAP Explanation ---
Calculating SHAP values for the test set... This may take some time.


  0%|          | 0/1462 [00:00<?, ?it/s]

Explaining predictions for class: juvenile


AssertionError: The shape of the shap_values matrix does not match the shape of the provided data matrix.

<Figure size 1000x600 with 0 Axes>